# data_normalization.ipynb

---

This block is used to convert data from ```generate-data.py``` to the format for ```sde_model.ipynb```.

While the implementation could have been done at the data generation level, this acts as an interoperable bridge betwen Edgar Fernández's code and Lucas Berredo's.

In [ ]:
import pandas as pd
import numpy as np
import torch
from scipy.interpolate import interp1d

csv_path = "../Datasets/dataset_trayectorias_completas.csv"
output_path = "../Datasets/flight_data.pt"
test_output_path = "../Datasets/flight_data_test.pt"

def preprocess_and_save_flights(csv_path="dataset_trayectorias_completas.csv", output_path="flight_data.pt", test_output_path="flight_data_test.pt", num_steps=200):
    df = pd.read_csv(csv_path)
    
    # 1. Define target features
    # Mapping: [Fuel, Velocity, Altitude]
    features = ['fuel_burnt_kg', 'vel_mps', 'baro_altitude']
    
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df[features] = df[features].interpolate(method="linear").bfill().ffill()

    # 2. Global Normalization (Min-Max)
    stats = {}
    for feat in features:
        f_min, f_max = df[feat].min(), df[feat].max()
        if f_max == f_min:
            df[feat] = 0.0
        else:
            df[feat] = (df[feat] - f_min) / (f_max - f_min)
        stats[feat] = {'min': float(f_min), 'max': float(f_max)}
    
    # 3. Group by Flight and Resample
    # Using 'icao24' + 'callsign' to identify unique flight trajectories
    df['flight_id'] = df['icao24'] + "_" + df['callsign']
    unique_flights = df['flight_id'].unique()
    
    all_flights_data = []
    
    for fid in unique_flights:
        flight_df = df[df['flight_id'] == fid].sort_values('time')
        
        # Normalize time for this flight to [0, 1] for interpolation
        t_orig = flight_df['time'].values
        t_norm = (t_orig - t_orig.min()) / (t_orig.max() - t_orig.min())
        
        # Target uniform time grid
        t_new = np.linspace(0, 1, num_steps)
        
        # Interpolate each feature to match the new grid
        resampled_features = []
        for feat in features:
            f_interp = interp1d(t_norm, flight_df[feat].values, kind='linear')
            resampled_features.append(f_interp(t_new))
        
        # Stack features: [num_steps, 3]
        all_flights_data.append(np.stack(resampled_features, axis=-1))
    
    test_flight_data = all_flights_data.pop()   # We need at least one test

    # 4. Convert to PyTorch Tensor
    # Final Shape: [Batch, Time, Features] -> [97, 200, 3]
    train_tensor = torch.tensor(np.stack(all_flights_data), dtype=torch.float32)
    test_tensor = torch.tensor(test_flight_data, dtype=torch.float32).unsqueeze(0)

    # 5. Permute for torchsde: [Time, Batch, Features] -> [200, 97, 3]
    y_true_train = train_tensor.permute(1, 0, 2)
    y_true_test = test_tensor.permute(1, 0, 2)

    t = torch.linspace(0, 1, num_steps)
    
    train_payload = {'y_true': y_true_train, 't': t, 'stats': stats}
    torch.save(train_payload, output_path)
    test_payload = {'y_true': y_true_test, 't': t, 'stats': stats}
    torch.save(test_payload, test_output_path)
    
    print(f"Successfully saved TRAIN set: {len(all_flights_data)} flights, shape {list(y_true_train.shape)}")
    print(f"Successfully saved TEST set: 1 flight, shape {list(y_true_test.shape)}")


# Run the preprocessing
preprocess_and_save_flights(csv_path, output_path, test_output_path)

Successfully saved TRAIN set: 96 flights, shape [200, 96, 3]
Successfully saved TEST set: 1 flight, shape [200, 1, 3]


---

This block is used to convert data from ```generate-data.py``` to the format for ```route_optimizer.py```.

While the implementation could have been done at the data generation level, this acts as an interoperable bridge betwen Edgar Vigil's code and Juan Boudón's.

In [ ]:
import pandas as pd
import numpy as np

imported_df = pd.read_csv("../Datasets/dataset_trayectorias_completas.csv")

# We need to create a tensor, serialized, with columns "Fuel", "Velocity", "Altitude", "Weather"